In [54]:
import csv
import folium
import json
import math
import os
import sys
import time
import warnings
from datetime import datetime, timedelta, date
from pathlib import Path
import branca.colormap as bcm
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import openeo
import rasterio
from dotenv import load_dotenv
from rasterio.crs import CRS as RIO_CRS
from rasterio.mask import mask as rio_mask
from rasterio.transform import from_bounds, array_bounds
from rasterio.warp import reproject, Resampling, transform_bounds
from shapely.geometry import mapping, shape
from sentinelhub import (
    BBox,
    CRS,
    DataCollection,
    MimeType,
    SHConfig,
    SentinelHubRequest,
    bbox_to_dimensions,
    SentinelHubCatalog
)
from sentinelhub.exceptions import DownloadFailedException, SHRateLimitWarning
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session
from json import JSONDecodeError

warnings.filterwarnings("ignore", category=SHRateLimitWarning)
load_dotenv()
# %pip install -r requirements.txt

True

In [55]:
# === GLOBAL ===
BASE_DIR = Path.cwd()
DATA_NAME = "data"
DATA_DIR = BASE_DIR / DATA_NAME
DO_PIPELINE = True

# === SECTION 5 ===
EXPORT_DIR = DATA_DIR / "export_consolidated"

LAKES = {
    "Atitlan":   DATA_DIR / "Lago_Atitlan.geojson",
    "Amatitlan": DATA_DIR / "Lago_Amatitlan.geojson",
}

STACKS = {
    ("Atitlan","CYANO"):     (EXPORT_DIR / "Atitlan_CYANO_stack.tif",     EXPORT_DIR / "Atitlan_CYANO_bands_dates.csv"),
    ("Atitlan","NDVI"):      (EXPORT_DIR / "Atitlan_NDVI_stack.tif",      EXPORT_DIR / "Atitlan_NDVI_bands_dates.csv"),
    ("Atitlan","NDWI"):      (EXPORT_DIR / "Atitlan_NDWI_stack.tif",      EXPORT_DIR / "Atitlan_NDWI_bands_dates.csv"),
    ("Amatitlan","CYANO"):   (EXPORT_DIR / "Amatitlan_CYANO_stack.tif",   EXPORT_DIR / "Amatitlan_CYANO_bands_dates.csv"),
    ("Amatitlan","NDVI"):    (EXPORT_DIR / "Amatitlan_NDVI_stack.tif",    EXPORT_DIR / "Amatitlan_NDVI_bands_dates.csv"),
    ("Amatitlan","NDWI"):    (EXPORT_DIR / "Amatitlan_NDWI_stack.tif",    EXPORT_DIR / "Amatitlan_NDWI_bands_dates.csv"),
}


# === SECTION 10: TIME SERIES FORECAST (CYANO) ===
RANDOM_SEED = 42
TS_FREQ = "7D"                 # frecuencia de re-muestreo (ej. 'D','7D','MS')
FORECAST_HORIZON = 30          # días a predecir
TRAIN_TEST_SPLIT_DATE = "2025-07-15"  # corte temporal para validación

# Rutas a series agregadas (ya calculadas en la parte 1)
TS_SERIES = {
    "Atitlan":   DATA_DIR / "analysis" / "Atitlan_CYANO_series.csv",
    "Amatitlan": DATA_DIR / "analysis" / "Amatitlan_CYANO_series.csv",
}

# Columna(s) esperadas en los CSV
TS_COL_DATE = "date"
TS_COL_VALUE = "cyano_index"   # p.ej. mean/p90 según definiste (CYANO_METRIC)

# Modelos a intentar
TS_MODELS = {
    "SARIMA":  {"use": True,  "order": (1,1,1), "seasonal_order": (0,1,1,4)},  # s=4 (semanal ~28d aprox)
    "PROPHET": {"use": False, "yearly_seasonality": False, "weekly_seasonality": True},
    "LSTM":    {"use": False, "lookback": 8, "epochs": 40, "batch_size": 32, "units": 64, "dropout": 0.2},
}

# Exógenas opcionales para TS (si ya tienes NDVI/NDWI promedio sincronizado por fecha)
TS_EXOG = {
    "use": True,
    "paths": {
        "Atitlan": {
            "NDVI": DATA_DIR / "analysis" / "Atitlan_NDVI_series.csv",
            "NDWI": DATA_DIR / "analysis" / "Atitlan_NDWI_series.csv",
        },
        "Amatitlan": {
            "NDVI": DATA_DIR / "analysis" / "Amatitlan_NDVI_series.csv",
            "NDWI": DATA_DIR / "analysis" / "Amatitlan_NDWI_series.csv",
        },
    },
    "date_col": "date",
    "value_cols": ["ndvi", "ndwi"],   # ajusta a los nombres reales
}

TS_OUTDIR = DATA_DIR / "forecast"
TS_OUTDIR.mkdir(parents=True, exist_ok=True)
TS_SAVE_COMPONENTS = True  # guardar gráficos/diagnósticos


# === SECTION 11: PIXEL-LEVEL CLASSIFICATION (PRESENCIA/NO) ===
# Umbral para definir etiqueta a partir del índice CYANO (puedes usar CYANO_THRESH de antes)
CYANO_PRESENCE_THRESHOLD = 0.10

# Muestreo espacial para entrenamiento
PIXEL_SAMPLING = {
    "strategy": "stratified",     # 'random'|'grid'|'stratified'
    "per_class": 20000,           # píxeles por clase (balanceado)
    "min_distance_m": 0,          # 0 = sin restricción
}

# Variables predictoras por píxel (derivables de los stacks)
PIXEL_FEATURES = ["CYANO", "NDVI", "NDWI", "x", "y", "month", "doy"]

# Stacks y calendario por lago (reusa tus STACKS/STACKS2)
CLASSIF_INPUT = {
    "Atitlan": {
        "stack_paths": {
            "CYANO": STACKS[("Atitlan","CYANO")][0],
            "NDVI":  STACKS[("Atitlan","NDVI")][0],
            "NDWI":  STACKS[("Atitlan","NDWI")][0],
        },
        "dates_csv": STACKS[("Atitlan","CYANO")][1],  # CSV con fechas por banda
        "mask_geojson": LAKES["Atitlan"],
    },
    "Amatitlan": {
        "stack_paths": {
            "CYANO": STACKS[("Amatitlan","CYANO")][0],
            "NDVI":  STACKS[("Amatitlan","NDVI")][0],
            "NDWI":  STACKS[("Amatitlan","NDWI")][0],
        },
        "dates_csv": STACKS[("Amatitlan","CYANO")][1],
        "mask_geojson": LAKES["Amatitlan"],
    },
}

# Modelos de clasificación
CLF_MODELS = {
    "LogReg":  {"use": True,  "C": 1.0, "class_weight": "balanced", "max_iter": 200},
    "RF":      {"use": True,  "n_estimators": 300, "max_depth": None, "n_jobs": -1, "class_weight": "balanced"},
    "XGB":     {"use": False, "n_estimators": 400, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8},
}

CLF_TRAIN_VAL = {
    "val_split": 0.2,        # split aleatorio si no se usa split temporal
    "by_date": False,        # True para separar por fechas (generaliza a nuevas fechas)
    "metrics": ["f1", "roc_auc", "precision", "recall"],
}

CLF_OUTDIR = DATA_DIR / "classification"
CLF_OUTDIR.mkdir(parents=True, exist_ok=True)


# === SECTION 12: HYBRID MODEL (FORECAST -> CLASSIFY) ===
HYBRID = {
    # 1) Predecir CYANO futuro usando la config de SECTION 10
    "use_ts_forecast": True,
    "forecast_horizon_days": FORECAST_HORIZON,

    # 2) Variables externas adicionales (opcional)
    "external_vars": {
        "use": True,
        "sources": {
            # coloca aquí rutas a CSV rasters agregados por fecha o por píxel
            # ejemplos (ajusta nombres reales):
            "temperature_surface": DATA_DIR / "external" / "temp_daily.csv",
            "precipitation":       DATA_DIR / "external" / "prcp_daily.csv",
            "urban_index":         DATA_DIR / "external" / "urban_index.tif",
        },
        "date_col": "date",
        "per_pixel": ["urban_index"],  # variables espaciales estáticas
        "per_date":  ["temperature_surface", "precipitation"],  # variables temporales
    },

    # 3) Clasificador final (usa índice CYANO predicho + features extra)
    "clf_base": "RF",  # 'RF'|'XGB'|'LogReg' — se tomará de CLF_MODELS
    "label_threshold": CYANO_PRESENCE_THRESHOLD,  # para convertir índice predicho a etiqueta
    "metrics": ["f1", "roc_auc"],
}

HYBRID_OUTDIR = DATA_DIR / "hybrid"
HYBRID_OUTDIR.mkdir(parents=True, exist_ok=True)


# === COMMON OUTPUTS ===
SAVE_PRED_MAPS = True        # guardar mapas de probabilidad/etiqueta
SAVE_RASTERS = True          # exportar GTiff con probabilidad por píxel
SAVE_REPORTS = True          # guardar JSON/CSV con métricas y params